# 🏢 Telecom Customer Churn — End-to-End BI Pipeline
**Author:** Haseeb Waqas  
**Stack:** Python (ETL) → SQLite (SQL) → Power BI (Dashboard)  
**Dataset:** IBM Telco Customer Churn (7,043 records)

---

## Pipeline Architecture
```
┌─────────────────────────────────────────────────────────────┐
│                    BI PIPELINE OVERVIEW                      │
│                                                              │
│  [Excel File]  [JSON API]                                    │
│       ↓              ↓                                       │
│    ┌──────────────────────┐                                  │
│    │   ETL Pipeline       │  ← Step 2 (replaces SSIS)        │
│    │  Extract→Transform   │                                  │
│    │  →Load to SQLite     │                                  │
│    └──────────┬───────────┘                                  │
│               ↓                                              │
│    ┌──────────────────────┐                                  │
│    │   SQL Analysis       │  ← Step 3                        │
│    │  Clean + Aggregate   │                                  │
│    └──────────┬───────────┘                                  │
│               ↓                                              │
│    ┌──────────────────────┐                                  │
│    │   Power BI Dashboard │  ← Step 4                        │
│    │  KPIs + Charts       │                                  │
│    └──────────────────────┘                                  │
└─────────────────────────────────────────────────────────────┘
```


## Step 1 — Data Sourcing

In [ ]:
%run ../pipeline/step1_data_sourcing.py

## Step 2 — ETL Pipeline

In [ ]:
%run ../pipeline/step2_etl_pipeline.py

## Step 3 — SQL Analysis

In [ ]:
%run ../pipeline/step3_sql_analysis.py

## Step 4 — Verify Output Files

In [ ]:
import pandas as pd
import sqlite3
import os

print('── Output Files ──')
files = [
    '../data/raw/source_a_demographics.xlsx',
    '../data/raw/source_b_billing_api.json',
    '../data/telecom_churn.db',
    '../data/telco_master_powerbi.csv',
    '../data/sql_analysis_results.xlsx',
]
for f in files:
    exists = '✅' if os.path.exists(f) else '❌'
    print(f'   {exists} {f}')

print('\n── Database Tables ──')
conn = sqlite3.connect('../data/telecom_churn.db')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for t in tables['name']:
    n = pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn)['n'][0]
    print(f'   ✅ {t}: {n:,} rows')
conn.close()

print('\n── Power BI Input File ──')
df = pd.read_csv('../data/telco_master_powerbi.csv')
print(f'   Rows    : {len(df):,}')
print(f'   Columns : {df.shape[1]}')
print(f'   Columns : {list(df.columns)}')

## Step 5 — Key Business Findings

In [ ]:
conn = sqlite3.connect('../data/telecom_churn.db')

kpis = pd.read_sql("""
    SELECT
        COUNT(*)                          AS Total_Customers,
        SUM(Churn_Binary)                 AS Churned,
        ROUND(AVG(Churn_Binary)*100,2)    AS Churn_Rate_Pct,
        ROUND(AVG(MonthlyCharges),2)      AS Avg_Monthly_Charge,
        ROUND(SUM(TotalCharges),0)        AS Total_Revenue,
        ROUND(SUM(CASE WHEN Churn_Binary=1
              THEN MonthlyCharges ELSE 0 END),0) AS Monthly_Revenue_At_Risk
    FROM fact_customers
""", conn)

print('='*55)
print('EXECUTIVE SUMMARY')
print('='*55)
for col in kpis.columns:
    val = kpis[col][0]
    if 'Revenue' in col or 'Charge' in col:
        print(f'   {col:<30}: ${val:,.0f}')
    elif 'Pct' in col or 'Rate' in col:
        print(f'   {col:<30}: {val}%')
    else:
        print(f'   {col:<30}: {val:,}')

conn.close()

print('\n💡 BUSINESS RECOMMENDATIONS')
recs = [
    'Month-to-month customers churn at 53% — offer loyalty discounts at 3-month mark',
    'New customers (0-12 months) churn at 51% — implement onboarding check-in program',
    'Fiber optic users churn most despite premium pricing — review service quality',
    'Electronic check users have highest churn — incentivize auto-pay enrollment',
]
for i, r in enumerate(recs, 1):
    print(f'   {i}. {r}')

## Step 6 — Next Step

✅ Pipeline complete. Now:

1. Open **Power BI Desktop**
2. Load `data/telco_master_powerbi.csv`
3. Follow `dashboard/POWERBI_GUIDE.md` to build the dashboard
4. Publish and paste your link into `README.md`
